# Problem Set 2
### Yuqi Gu

## 1 Hospital admission & quality of service

### Question 1
#### (a)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
hdata = pd.read_csv('health_data.csv')
hdata.head()

,patient_id,hospital_id,admin_year,patient_died_dummy,startage,female_dummy
0,1,D,2003,0,81,0
1,2,H,2003,1,67,0
2,3,A,2003,0,54,0
3,4,E,2003,0,81,0
4,5,H,2003,0,69,0


In [ ]:
model1  = smf.ols(formula = 'patient_died_dummy ~ hospital_id',data=hdata)
result1 = model1.fit()
print(result1.summary())

                            OLS Regression Results                            
Dep. Variable:     patient_died_dummy   R-squared:                       0.042
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     119.3
Date:                Mon, 10 Feb 2025   Prob (F-statistic):          1.75e-220
Time:                        03:19:53   Log-Likelihood:                -7416.5
No. Observations:               24480   AIC:                         1.485e+04
Df Residuals:                   24470   BIC:                         1.493e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.0970      0.006  

Intercept(0.097) represents the average mortality rate for the hospital A.  The coefficient on the dummy for hospital D(0.1882) indicates the difference in the mortality rate between hospital D and hospital A. In other words, the mortality rate of hospital D is 0.1882 higher than the mortality rate of hospital A.

#### (b)

In [ ]:
result1.params['hospital_id[T.D]'] - result1.params['hospital_id[T.E]']

0.24139049162000606

The difference between the mortality rates at hospitals D and E is 0.24139049162000606. In other words, the mortality rate of hospital D is 0.24139049162000606 higher than the mortality rate of hospital E.  This is because the result1.params['hospital_id[T.D]'] represents the difference in the mortality rate between hospital D and hospital A and result1.params['hospital_id[T.E]'] represents the difference in the mortality rate between hospital E and hospital A.

### Question 2
#### (a)

In [ ]:
hdata_AB = hdata[hdata['hospital_id'].isin(['A', 'B'])]
model2  = smf.ols(formula = 'patient_died_dummy ~ hospital_id',data=hdata_AB)
result2 = model2.fit()
print(result2.summary())


                            OLS Regression Results                            
Dep. Variable:     patient_died_dummy   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.9377
Date:                Mon, 10 Feb 2025   Prob (F-statistic):              0.333
Time:                        03:19:56   Log-Likelihood:                -1446.8
No. Observations:                6611   AIC:                             2898.
Df Residuals:                    6609   BIC:                             2911.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.0970      0.005  

Patients are not randomly assigned to hospitals. The patients go to the hospitals by their wishes. Some people may directly go to the hospital which is the nearest, some people may go to the famous hospital and some people may go to the hospital they have friends who work there, etc. There are many factors influence if the patient goes to which hospital. Also, the severity of illness, financial conditions, or other factors may influence which hospital a patient goes to. Furthermore, different hospital has different equipments, staff expertise, and histories, etc. This makes the direct comparisons misleading. Furthermore, some other variables like the health conditions of the patients may drive diverse hospital choice and mortality risk. Moreover, there are confounding factors such as if the patients in hospital B tend to have more severe conditions, then the mortality rate might be higher even if the hospital B provides same or better care and cure methods. Also, a hospital's reputation might also influence the choices of patients and some high-risk patients are more willing to go to the hospital with higher reputation. Thus, the difference in mortality rate of these two hospitals might be due to the differences in patient conditions rather than the choice of choosing these two hospitals.
Thus, without randomization or proper controls, we cannot infer that moving a patient from Hospital A to B would change their probability of dying.

#### (b)
If we assume that patients with higher risks are more likely to go to the hospital B, then the difference in mortality between hospitals are overestimated. This is because the model is just included hospital_id as the independent variable. Then the effects such as the patients with higher risks and more severe conditions are more likely to go to the hospital B might increase the mortality rate of hospital B and this is attributed into the effects of the hospital_id in this model. Thus, in this case the difference in mortality between hospitals are overestimated. We might also consider that if
hospital B is much better equipped, it might lower mortality despite receiving many sicker patients, leading to an underestimated effect. This bias makes direct comparisons unreliable without additional control variables.

#### (c)

In [ ]:
model3 = smf.ols(formula = 'patient_died_dummy ~ hospital_id+startage+female_dummy+admin_year',data=hdata_AB)
result3 = model3.fit()
print(result3.summary())


                            OLS Regression Results                            
Dep. Variable:     patient_died_dummy   R-squared:                       0.063
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     110.7
Date:                Mon, 10 Feb 2025   Prob (F-statistic):           1.66e-91
Time:                        03:45:12   Log-Likelihood:                -1232.7
No. Observations:                6611   AIC:                             2475.
Df Residuals:                    6606   BIC:                             2509.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -2.4995      5.161  

To move closer to a causal estimate, we need to control for some variables such as the startage, gender, and admin_year. We notice that the coefficient of hospital_id[T.B] becomes larger from 0.0072 to 0.0114. This indicates that the estimated difference in mortality rate between Hospital B and A increases after controlling for these three patient characteristics. This increase suggests that after accounting for some other variables, it is possible that Hospital B has higher mortality rate than Hospital A but it is not statistically significant since p-value (0.114) is larger than 0.05. The reasons why we include the specific set of variables are we hope to control for these patient characteristics variables and see the difference in the mortality rate between B and A more clearly in order to at least get closer to a causal estimate. We also observe that the p-value of startage (0.018) and female_dummy (0.000) is smaller than 0.05 and so these two variables are statistical significant. Thus, it is possible that these two influence the mortality rate for the hospitals. Also, older people might be more vulnerable to disease and it is possible that female might be more vulnerable to the disease. Furthermore, the global conditions which indicated by the admin_year might also influence the mortality rate. The medical technology, disease situations, and other global factors might affect the mortality rate. Thus, these three variables might be included in order to get closer to a causal estimate. By observing the p-values, the p-values for startage and female_dummy are smaller than 0.05 but admin_year is not. Thus, startage and female_dummy are statistically significant.

## 2 Demand estimation
### Question 1

In [ ]:
ddata = pd.read_csv('demand_data.csv')
ddata

,vendor_id,week,summer_dummy,price,sales
0,1,1,0,2.0,8788.7383
1,1,2,0,3.0,8937.9863
2,1,3,0,3.0,8740.1777
3,1,4,0,3.0,8757.1338
4,1,5,0,3.0,8739.6104
...,...,...,...,...,...
5195,100,48,0,2.0,8930.7900
5196,100,49,0,1.5,8583.6533
5197,100,50,0,1.5,8964.9170
5198,100,51,0,2.5,8811.0869


In [ ]:
ddata_vendor1 = ddata[ddata['vendor_id'] == 1]

model_sp = smf.ols(formula = 'sales ~ price',data=ddata_vendor1)
result_sp = model_sp.fit()
print(result_sp.summary())


                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.006
Model:                            OLS   Adj. R-squared:                 -0.013
Method:                 Least Squares   F-statistic:                    0.3250
Date:                Mon, 10 Feb 2025   Prob (F-statistic):              0.571
Time:                        03:45:24   Log-Likelihood:                -360.33
No. Observations:                  52   AIC:                             724.7
Df Residuals:                      50   BIC:                             728.6
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   8983.8227    145.437     61.771      0.0

In [ ]:
model_sp_summer = smf.ols(formula = 'sales ~ price + summer_dummy',data=ddata_vendor1)
result_sp_summer = model_sp_summer.fit()
print(result_sp_summer.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.318
Model:                            OLS   Adj. R-squared:                  0.290
Method:                 Least Squares   F-statistic:                     11.42
Date:                Mon, 03 Feb 2025   Prob (F-statistic):           8.49e-05
Time:                        19:24:14   Log-Likelihood:                -350.56
No. Observations:                  52   AIC:                             707.1
Df Residuals:                      49   BIC:                             713.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     9177.5500    128.432     71.458   

We observe that for the regression model of sales on price,  the price coefficient is -31.2310. For the regression model of sales on price and summer_dummy, we find the price coefficient is -141.1887 and the coefficient for summer_dummy is 358.5012. Also, the p-value for the coefficient of price is 0.571 in the first model and it becomes 0.008 in the second model. This indicates that the price is statistically significant in the second model. After adding the summer dummy, the negative effect of price on sales becomes larger.

The omitted variable bias happens when a statistical model leaves out one or more relevant variables. The bias results in the model attributing the effect of the missing variables to those that were included. Based on the omitted variable bias formula, we know that the difference between the price coefficient when there is no summer dummy included and the price coefficient when there is summer dummy included is caused by the bias. When the summer dummy is not included, the effect of summer dummy is attributed to the price. In this case, the summy dummy coefficient is positive and it means the summer dummy has positive effects on sales. When there is no summer dummy in the model, this positive effect is included into the coeffcient of price, making it become more positive than the true effect of the price on sales. If summer are related to both sales and price, omitting it from the model causes the price coefficient to be biased toward zero, which means the true effect of the price on sales is distorted. Once summer dummy is included, price's effect is properly accounted for,making the coefficient more negative and the p-value becomes much smaller.

#### Question 2

In [ ]:
ddata_vendor2 = ddata[ddata['vendor_id'] == 2]

model_sp2 = smf.ols(formula = 'sales ~ price',data=ddata_vendor2)
result_sp2 = model_sp2.fit()
print(result_sp2.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.133
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     7.684
Date:                Mon, 10 Feb 2025   Prob (F-statistic):            0.00781
Time:                        03:49:33   Log-Likelihood:                -359.10
No. Observations:                  52   AIC:                             722.2
Df Residuals:                      50   BIC:                             726.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   8411.1748    219.545     38.312      0.0

In [ ]:
model_sp_summer2 = smf.ols(formula = 'sales ~ price + summer_dummy',data=ddata_vendor2)
result_sp_summer2 = model_sp_summer2.fit()
print(result_sp_summer2.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.133
Model:                            OLS   Adj. R-squared:                  0.116
Method:                 Least Squares   F-statistic:                     7.684
Date:                Mon, 10 Feb 2025   Prob (F-statistic):            0.00781
Time:                        03:49:36   Log-Likelihood:                -359.10
No. Observations:                  52   AIC:                             722.2
Df Residuals:                      50   BIC:                             726.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     2105.3159     29.848     70.534   

Based on the regression result, we find the smallest eigenvalue is 2.61e-30. This might indicate that there are strong multicollinearity problems or that the design matrix is singular. Also, we observe that the standard errors of the coefficients are large (75.986 for summer_dummy), which is another sign of multicollinearity.

The reasons that Price and Summer Dummy are highly correlated might be the vendor 2 only changes prices in summer and keeps price constant for other times, which would cause the price and summer_dummy are nearly collinear. Thus, the effects of price or the summer dummy on sales are hard to distinguish.


#### Question 3

If one of the vendors did not systematically charge higher or lower prices in summer, the relationship between price and summer_dummy would be weak or nonexistent for this vendor.
Thus, if we repeat the analysis we just did for vendors 1 and 2 for this vendor, without the summer dummy, we will observe that the coefficient for the price remain unbiased and the coefficient will capture the true relationship between sales and price without confounding from summer dummy. Also, its precision would be high since the standard error of price would be low because price and summer dummy are not correlated.
For the model with the summer dummy, the coefficient for price will remain stable because there is no collinearity between summer dummy and price. This means there is no large change for the coefficient of price when the summer dummy is added into the model. So the inclusion of summer dummy would not change the price coefficient a lot but the summer dummy might capture some variations in sales but it will not distort the coefficient for price. The precision will be high because the inclusion of summer dummy should not introduce multicollinearity since the summer dummy and price are not highly correlated. And so the standard error would be low and the precision is high.
